# Official GPT Training on Google Colab

This notebook acts as an orchestration layer to seamlessly run the platform-independent GPT codebase on Google Colab without modifying the core logic.

**Workflow:**
1. Pull the latest code from GitHub.
2. Mount Google Drive to save checkpoints.
3. Train locally on the Colab T4/L4 GPU.
4. Save results to Drive.

## 1. Setup GitHub Repository

We clone the official repository to ensure we are always running the latest version of the code.

In [ ]:
import os

# Clone the repository if it doesn't exist, otherwise pull the latest changes
if not os.path.exists('LLM'):
    !git clone https://github.com/Harshkumar2306/LLM.git
    %cd LLM
else:
    %cd LLM
    !git pull

# Install dependencies
!pip install -r requirements.txt

# Verify GPU
!nvidia-smi

## 2. Mount Google Drive

Mounting Google Drive allows us to save checkpoints securely. If Colab disconnects, you can simply reconnect, re-run this notebook, and it will automatically resume from the latest checkpoint.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/AtlasLM'
    print(f"Google Drive mounted! Training will save to {OUT_DIR}")
except ImportError:
    OUT_DIR = 'runs/colab_run'
    print(f"Not running in Colab (or Drive not mounted). Checkpoints will save to local {OUT_DIR}")

## 3. Dataset Preparation

If you haven't prepared the Tiny Shakespeare dataset yet, run this cell to download and encode it into `.bin` files.

In [ ]:
!python scripts/prepare_data.py

## 4. Training (With Auto-Resume)

By specifying `--out_dir {OUT_DIR}`, the script will automatically resume training from `latest.pt` if your Colab session reconnects!

In [ ]:
!python scripts/train.py \
    --train_bin data/train.bin \
    --val_bin data/val.bin \
    --out_dir {OUT_DIR} \
    --vocab_size 50257 \
    --d_model 256 \
    --n_heads 8 \
    --n_layers 6 \
    --context_length 256 \
    --batch_size 64 \
    --max_iters 5000 \
    --eval_interval 500 \
    --learning_rate 6e-4

## 5. Text Generation

Test your trained model!

In [ ]:
import os
best_checkpoint = os.path.join(OUT_DIR, "best.pt")

!python scripts/generate.py \
    --checkpoint {best_checkpoint} \
    --prompt "ROMEO:" \
    --max_new_tokens 200 \
    --temperature 0.8 \
    --use_cache

## 6. Push Results to GitHub (Optional)

Once training is complete, you can push the logs and reports back to your GitHub repository to track experiments.

In [ ]:
# !git config --global user.email "you@example.com"
# !git config --global user.name "Your Name"
# !git add {OUT_DIR}/report.txt {OUT_DIR}/metrics.jsonl
# !git commit -m "Colab Experiment Results"
# !git push origin main